Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC -- Metabase / Trino — paste entire file into Native Query.
# MAGIC -- Returns a rolling 15-day cohort of CONNECTED tickets
# MAGIC -- 2026-05-29: includes disbursed-but-open tickets
# MAGIC
# MAGIC WITH cohort AS (
# MAGIC     SELECT
# MAGIC         t.customer_id,
# MAGIC         t.canonical_ticket_id,
# MAGIC         t.first_ticket_created_date,
# MAGIC         t.disposition_reason,
# MAGIC         t.disposition_sub_reason,
# MAGIC         t.total_calls,
# MAGIC         t.offer_amt,
# MAGIC         t.disbursal_flag,
# MAGIC         t.rejection_flag,
# MAGIC         t.chat_ticket_state,
# MAGIC         t.pa_od_tag,
# MAGIC         t.chat_assigned_agent,
# MAGIC         t.agents_sequential,
# MAGIC         t.call_creation_timestamps,
# MAGIC         t.sticky_agent_name,
# MAGIC         ROW_NUMBER() OVER (
# MAGIC             PARTITION BY t.customer_id
# MAGIC             ORDER BY t.first_ticket_created_date DESC NULLS LAST
# MAGIC         ) AS rn
# MAGIC     FROM corp_analytics.pl_apex_canonical_tickets_tbl t
# MAGIC     WHERE DATE(t.first_ticket_created_date)
# MAGIC           BETWEEN CURRENT_DATE - INTERVAL '15' DAY AND CURRENT_DATE
# MAGIC       AND t.chat_ticket_state NOT IN ('FORCE_CLOSED', 'RESOLVED')
# MAGIC       AND COALESCE(t.disposition_reason, '') NOT IN ('', 'Not Connected')
# MAGIC ),
# MAGIC
# MAGIC apex_tickets AS (
# MAGIC     SELECT *
# MAGIC     FROM cohort
# MAGIC     WHERE rn = 1
# MAGIC ),
# MAGIC
# MAGIC cust_bsd AS (
# MAGIC     SELECT
# MAGIC         o.customer_reference_id AS customer_id,
# MAGIC         TRY_CAST(s.bsdv1_retrain_tier AS INTEGER) AS bsd_tier
# MAGIC     FROM corp_analytics.customer_last_approved_offer_tbl o
# MAGIC     JOIN corp_analytics.pl_uw_scores_tiers_tbl s
# MAGIC         ON s.loan_offer_reference_id = o.loan_offer_reference_id
# MAGIC     WHERE TRY_CAST(s.bsdv1_retrain_tier AS INTEGER) BETWEEN 1 AND 10
# MAGIC       AND CAST(o.customer_reference_id AS VARCHAR)
# MAGIC             IN (SELECT customer_id FROM apex_tickets)
# MAGIC ),
# MAGIC
# MAGIC o2d_stages AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         DATE(CAST(event_date AS timestamp)) AS event_dt,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'OFFER_NEXT'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS offer_next_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'PENNY'
# MAGIC                      AND COALESCE(is_silent_penny, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS penny_init_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'PENNY'
# MAGIC                      AND status = 'SUCCESS'
# MAGIC                      AND COALESCE(is_silent_penny, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS penny_success_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'MANDATE'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS mandate_init_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'MANDATE'
# MAGIC                      AND status IN ('SUCCESS', 'ACTIVE', 'REGISTERED')
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS mandate_success_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'KYC'
# MAGIC                      AND COALESCE(is_kyc_replication, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS kyc_success_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'LOAN_SUMMARY_NEXT'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS loan_summary_next_ts
# MAGIC
# MAGIC     FROM corp_analytics.pl_raw_stage_tbl
# MAGIC     WHERE CAST(event_date AS timestamp) >= TIMESTAMP '2026-02-01 00:00:00'
# MAGIC       AND stage_tag IN (
# MAGIC             'OFFER_NEXT',
# MAGIC             'PENNY',
# MAGIC             'MANDATE',
# MAGIC             'KYC',
# MAGIC             'LOAN_SUMMARY_NEXT'
# MAGIC       )
# MAGIC       AND customer_id IN (SELECT customer_id FROM apex_tickets)
# MAGIC
# MAGIC     GROUP BY 1, 2
# MAGIC ),
# MAGIC
# MAGIC sales AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         CAST(disbursed_date AS date) AS disbursal_date
# MAGIC     FROM corp_analytics.pl_sales_fct_tbl
# MAGIC     WHERE CAST(disbursed_date AS date) >= DATE '2026-02-01'
# MAGIC       AND customer_id IN (SELECT customer_id FROM apex_tickets)
# MAGIC ),
# MAGIC
# MAGIC o2d_post_ticket AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         s.offer_next_ts,
# MAGIC         s.penny_init_actual_ts,
# MAGIC         s.penny_success_actual_ts,
# MAGIC         s.mandate_init_ts,
# MAGIC         s.mandate_success_ts,
# MAGIC         s.kyc_success_actual_ts,
# MAGIC         s.loan_summary_next_ts
# MAGIC     FROM apex_tickets a
# MAGIC     JOIN o2d_stages s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC        AND CAST(s.event_dt AS date)
# MAGIC             >= CAST(a.first_ticket_created_date AS date)
# MAGIC        AND CAST(s.event_dt AS date)
# MAGIC             <= CAST(a.first_ticket_created_date AS date) + INTERVAL '7' DAY
# MAGIC ),
# MAGIC
# MAGIC o2d_consolidated AS (
# MAGIC     SELECT
# MAGIC         canonical_ticket_id,
# MAGIC         MIN(offer_next_ts) AS offer_next_ts,
# MAGIC         MIN(penny_init_actual_ts) AS penny_init_actual_ts,
# MAGIC         MIN(penny_success_actual_ts) AS penny_success_actual_ts,
# MAGIC         MIN(mandate_init_ts) AS mandate_init_ts,
# MAGIC         MIN(mandate_success_ts) AS mandate_success_ts,
# MAGIC         MIN(kyc_success_actual_ts) AS kyc_success_actual_ts,
# MAGIC         MIN(loan_summary_next_ts) AS loan_summary_next_ts
# MAGIC     FROM o2d_post_ticket
# MAGIC     GROUP BY 1
# MAGIC ),
# MAGIC
# MAGIC disbursal_match AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         MIN(s.disbursal_date) AS funnel_disbursal_date
# MAGIC     FROM apex_tickets a
# MAGIC     JOIN sales s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC        AND CAST(s.disbursal_date AS date)
# MAGIC             >= CAST(a.first_ticket_created_date AS date)
# MAGIC        AND CAST(s.disbursal_date AS date)
# MAGIC             <= CAST(a.first_ticket_created_date AS date) + INTERVAL '7' DAY
# MAGIC     GROUP BY 1
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     a.customer_id,
# MAGIC     a.canonical_ticket_id,
# MAGIC     a.first_ticket_created_date,
# MAGIC
# MAGIC     DATE_DIFF(
# MAGIC         'day',
# MAGIC         CAST(a.first_ticket_created_date AS date),
# MAGIC         CURRENT_DATE
# MAGIC     ) AS days_in_pipeline,
# MAGIC
# MAGIC     a.disposition_reason,
# MAGIC     a.disposition_sub_reason,
# MAGIC     a.total_calls,
# MAGIC     a.offer_amt,
# MAGIC     a.disbursal_flag,
# MAGIC     a.rejection_flag,
# MAGIC     a.chat_ticket_state,
# MAGIC     a.pa_od_tag,
# MAGIC     a.chat_assigned_agent,
# MAGIC     a.agents_sequential,
# MAGIC     a.call_creation_timestamps,
# MAGIC
# MAGIC     cb.bsd_tier,
# MAGIC
# MAGIC     CASE
# MAGIC         WHEN d.funnel_disbursal_date IS NOT NULL
# MAGIC             THEN 'DISBURSED'
# MAGIC
# MAGIC         WHEN o.loan_summary_next_ts IS NOT NULL
# MAGIC             THEN 'LOAN_SUMMARY'
# MAGIC
# MAGIC         WHEN o.kyc_success_actual_ts IS NOT NULL
# MAGIC             THEN 'KYC_SUCCESS'
# MAGIC
# MAGIC         WHEN o.mandate_success_ts IS NOT NULL
# MAGIC             THEN 'MANDATE_SUCCESS'
# MAGIC
# MAGIC         WHEN o.mandate_init_ts IS NOT NULL
# MAGIC             THEN 'MANDATE_INIT'
# MAGIC
# MAGIC         WHEN o.penny_success_actual_ts IS NOT NULL
# MAGIC             THEN 'PENNY_SUCCESS'
# MAGIC
# MAGIC         WHEN o.penny_init_actual_ts IS NOT NULL
# MAGIC             THEN 'PENNY_INIT'
# MAGIC
# MAGIC         WHEN o.offer_next_ts IS NOT NULL
# MAGIC             THEN 'OFFER_NEXT'
# MAGIC
# MAGIC         ELSE 'OFFER_ONLY'
# MAGIC     END AS furthest_stage,
# MAGIC
# MAGIC     ELEMENT_AT(
# MAGIC         CAST(JSON_PARSE(a.agents_sequential) AS ARRAY(VARCHAR)),
# MAGIC         -1
# MAGIC     ) AS last_calling_agent,
# MAGIC
# MAGIC     a.sticky_agent_name
# MAGIC
# MAGIC FROM apex_tickets a
# MAGIC
# MAGIC LEFT JOIN cust_bsd cb
# MAGIC     ON cb.customer_id = a.customer_id
# MAGIC
# MAGIC LEFT JOIN o2d_consolidated o
# MAGIC     ON o.canonical_ticket_id = a.canonical_ticket_id
# MAGIC
# MAGIC LEFT JOIN disbursal_match d
# MAGIC     ON d.canonical_ticket_id = a.canonical_ticket_id
# MAGIC
# MAGIC ORDER BY
# MAGIC     a.first_ticket_created_date DESC,
# MAGIC     a.canonical_ticket_id
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1iWBV4WBiS_Gq1SYIxIzbjXb9bBEy0maZ5mV5I2g-lnY/edit"  # New V4

worksheet_name = "Updated call dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)